Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 26.9 ms, sys: 4.03 ms, total: 30.9 ms
Wall time: 30.4 ms
CPU times: user 10.8 ms, sys: 4.01 ms, total: 14.8 ms
Wall time: 14.7 ms
CPU times: user 3.61 ms, sys: 2 ms, total: 5.61 ms
Wall time: 5.62 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 150
alpha = 1
lr = 1e-4
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0
shared_dropout = 0
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.0001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [9]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfr = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        method=method, alpha=alpha,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfr.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfr.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312


🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 437.0244 | Val Loss: 490.6410 | Val Qini: 0.3808 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3808 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 638.7494 | Val Loss: 490.5783 | Val Qini: 0.3741 (patience: 1/20)EMA Trend: 0.3798 | (patience: 1/20)
Epoch 3/150 | Loss: 294.7515 | Val Loss: 490.5114 | Val Qini: 0.3491 (patience: 2/20)EMA Trend: 0.3752 | (patience: 2/20)
Epoch 4/150 | Loss: 424.6520 | Val Loss: 490.4352 | Val Qini: 0.3551 (patience: 3/20)EMA Trend: 0.3721 | (patience: 3/20)
Epoch 5/150 | Loss: 356.9093 | Val Loss: 490.3489 | Val Qini: 0.3431 (patience: 4/20)EMA Trend: 0.3678 | (patience: 4/20)
Epoch 6/150 | Loss: 328.5348 | Val Loss: 490.2509 | Val Qini: 0.3044 (patience: 5/20)EMA Trend: 0.3583 | (patien

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 332.0768 | Val Loss: 490.6268 | Val Qini: 0.1990 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.1990 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 598.3302 | Val Loss: 490.5742 | Val Qini: 0.1911 (patience: 1/20)EMA Trend: 0.1978 | (patience: 1/20)
Epoch 3/150 | Loss: 412.7926 | Val Loss: 490.5181 | Val Qini: 0.2404 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2042 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 470.3491 | Val Loss: 490.4562 | Val Qini: 0.3325 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2234 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 470.6924 | Val Loss: 490.3878 | Val Qini: 0.3871 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2480 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 166.3669 | Val Loss: 490.3112 | Val Qini: 0.4312 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2755 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 436.1776 | Val Loss: 490.2242 | Val Qini: 0.4402 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3002 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 336.4120 | Val Los

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 524.7335 | Val Loss: 490.6086 | Val Qini: 0.6930 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6930 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 231.2861 | Val Loss: 490.5493 | Val Qini: 0.6096 (patience: 1/20)EMA Trend: 0.6805 | (patience: 1/20)
Epoch 3/150 | Loss: 546.6509 | Val Loss: 490.4916 | Val Qini: 0.4455 (patience: 2/20)EMA Trend: 0.6452 | (patience: 2/20)
Epoch 4/150 | Loss: 527.4208 | Val Loss: 490.4306 | Val Qini: 0.3925 (patience: 3/20)EMA Trend: 0.6073 | (patience: 3/20)
Epoch 5/150 | Loss: 344.4892 | Val Loss: 490.3643 | Val Qini: 0.4457 (patience: 4/20)EMA Trend: 0.5831 | (patience: 4/20)
Epoch 6/150 | Loss: 289.6888 | Val Loss: 490.2887 | Val Qini: 0.4981 (patience: 5/20)EMA Trend: 0.5703 | (patience: 5/20)
Epoch 7/150 | Loss: 374.2393 | Val Loss: 490.2026 | Val Qini: 0.5869 ✓ above trend but not peak (patience: 6/20)EMA Trend: 0.5728 | ✓ above trend but not peak (patience: 6/20)
Epoch 8/150 | Loss: 597.4529 | Val Loss: 490.1060 | Val Qini: 0.6666 ✓ abo

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 293.1359 | Val Loss: 490.7155 | Val Qini: 0.5421 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5421 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 397.2821 | Val Loss: 490.6641 | Val Qini: 0.5715 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5465 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 411.0444 | Val Loss: 490.6092 | Val Qini: 0.6013 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5547 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 544.9208 | Val Loss: 490.5471 | Val Qini: 0.6849 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5742 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 461.2090 | Val Loss: 490.4780 | Val Qini: 0.7437 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5996 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 497.5440 | Val Loss: 490.3952 | Val Qini: 0.7463 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6216 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 464.1660 | Val Loss: 490.2956 | Val Qini: 0.7535 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6414 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
